#### Autora: Giulia Katherine Romas Tomazeli

# Bibliotecas

In [1]:
from pathlib import Path
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
import sys
import optuna

In [2]:
PROJECT_DIR = Path.cwd().parent
sys.path.append(str(PROJECT_DIR))

In [3]:
from configs.config import (
    RESULTS_DIR,
    IMAGE_SIZE,
    BATCH_SIZE,
    NUM_WORKERS,
    NUM_CLASSES,
    CLASS_NAMES,
    MEAN,
    STD,
    DEVICE,
    SEED,
    TRAIN_DIR
)

Device: cuda


In [4]:
from utils.dataset import NascentesDataset, samples_to_dataframe
from utils.spatial_kfold import (load_spatial_groups, create_spatial_folds)
from utils.optimization import objective

c:\Users\giuli\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.2)
  from scipy.sparse import csr_matrix, issparse


# Configurações

In [5]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [6]:
MODEL_NAME = "final_model"
N_SPLITS = 5

print(f"Modelo: {MODEL_NAME}")
print(f"Device: {DEVICE}")
print(f"Folds: {N_SPLITS}")

Modelo: final_model
Device: cuda
Folds: 5


In [7]:
SHAPEFILE_PATH = PROJECT_DIR / "data" / "amostras" / "amostra_final.shp"

In [8]:
result_dir_test = RESULTS_DIR / MODEL_NAME / "test"
result_dir_test.mkdir(parents=True, exist_ok=True)

# Dataset

In [9]:
train_dataset = NascentesDataset(root_dir=TRAIN_DIR, class_to_idx={"Preservada": 0, "Moderada": 1, "Baixa": 2, "Não Preservada": 3}, image_size=IMAGE_SIZE, mean=MEAN, std=STD)

In [10]:
samples_df = samples_to_dataframe(train_dataset)

print(f"Total de imagens: {len(samples_df)}")
samples_df.head()

Total de imagens: 1538


,path,label,fid
0,C:\Users\giuli\OneDrive\Área de Trabalho\INPE\...,0,fid_1004354
1,C:\Users\giuli\OneDrive\Área de Trabalho\INPE\...,0,fid_1004578
2,C:\Users\giuli\OneDrive\Área de Trabalho\INPE\...,0,fid_1016991
3,C:\Users\giuli\OneDrive\Área de Trabalho\INPE\...,0,fid_1019727
4,C:\Users\giuli\OneDrive\Área de Trabalho\INPE\...,0,fid_103701


In [11]:
dataset_fids = samples_df["fid"].tolist()

# Spatial K-Fold

In [12]:
spatial_groups = load_spatial_groups(shapefile_path=SHAPEFILE_PATH, dataset_fids=dataset_fids)

spatial_groups.head()

,fid,ugrhi_id,ugrhi_nome
0,1365920,20,Aguapeí
1,764554,20,Aguapeí
2,639552,20,Aguapeí
4,1894144,20,Aguapeí
5,1714024,20,Aguapeí


In [13]:
samples_with_groups, folds = create_spatial_folds(samples=samples_df, spatial_groups=spatial_groups, n_splits=N_SPLITS, random_state=SEED)

In [14]:
for fold in folds:
    print(f"Fold {fold['fold']}: treino={len(fold['train_idx'])}, validação={len(fold['val_idx'])}")

Fold 1: treino=1192, validação=346
Fold 2: treino=1265, validação=273
Fold 3: treino=1181, validação=357
Fold 4: treino=1258, validação=280
Fold 5: treino=1256, validação=282


## Verificação dos folds

In [15]:
for fold in folds:
    train_idx = fold["train_idx"]
    val_idx = fold["val_idx"]

    print(f"\n=-=-= Fold {fold['fold']} =-=-=")

    print("Treino:")
    print(samples_with_groups.iloc[train_idx]["label"].value_counts(normalize=True).sort_index())

    print("\nValidação:")
    print(samples_with_groups.iloc[val_idx]["label"].value_counts(normalize=True).sort_index())


=-=-= Fold 1 =-=-=
Treino:
label
0    0.387584
1    0.216443
2    0.248322
3    0.147651
Name: proportion, dtype: float64

Validação:
label
0    0.320809
1    0.190751
2    0.268786
3    0.219653
Name: proportion, dtype: float64

=-=-= Fold 2 =-=-=
Treino:
label
0    0.369960
1    0.207905
2    0.246640
3    0.175494
Name: proportion, dtype: float64

Validação:
label
0    0.384615
1    0.223443
2    0.282051
3    0.109890
Name: proportion, dtype: float64

=-=-= Fold 3 =-=-=
Treino:
label
0    0.351397
1    0.209145
2    0.266723
3    0.172735
Name: proportion, dtype: float64

Validação:
label
0    0.442577
1    0.215686
2    0.207283
3    0.134454
Name: proportion, dtype: float64

=-=-= Fold 4 =-=-=
Treino:
label
0    0.373609
1    0.206677
2    0.255962
3    0.163752
Name: proportion, dtype: float64

Validação:
label
0    0.367857
1    0.228571
2    0.239286
3    0.164286
Name: proportion, dtype: float64

=-=-= Fold 5 =-=-=
Treino:
label
0    0.379777
1    0.213376
2    0.247611
3   

In [16]:
for fold in folds:
    train_ugrhi = set(samples_with_groups.iloc[fold["train_idx"]]["ugrhi_id"])
    val_ugrhi = set(samples_with_groups.iloc[fold["val_idx"]]["ugrhi_id"])

    overlap = train_ugrhi.intersection(val_ugrhi)

    print(f"Fold {fold['fold']} - UGRHIs compartilhadas: {len(overlap)}")

Fold 1 - UGRHIs compartilhadas: 0
Fold 2 - UGRHIs compartilhadas: 0
Fold 3 - UGRHIs compartilhadas: 0
Fold 4 - UGRHIs compartilhadas: 0
Fold 5 - UGRHIs compartilhadas: 0


# Optuna

In [17]:
# Fold utilizado para otimização dos hiperparâmetros
optuna_fold = folds[0]

train_idx = optuna_fold["train_idx"]
val_idx = optuna_fold["val_idx"]

train_subset = Subset(train_dataset, train_idx)
val_subset = Subset(train_dataset, val_idx)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [18]:
study = optuna.create_study(direction="minimize", study_name="final_model")

study.optimize(
    lambda trial: objective(
        trial=trial,
        train_loader=train_loader,
        val_loader=val_loader,
        train_subset=train_subset,
        device=DEVICE,
        num_classes=NUM_CLASSES,
        class_names=CLASS_NAMES
    ),
    n_trials=30
)

print("Melhores hiperparâmetros:")
print(study.best_params)

print("Melhor val loss:")
print(study.best_value)

[I 2026-10-25 20:55:21,679] A new study created in memory with name: final_model
[I 2026-10-25 21:14:52,188] Trial 0 finished with value: 0.6027663431071133 and parameters: {'learning_rate': 0.0001283317520141486, 'weight_decay': 3.8202459370907955e-05, 'dropout': 0.528271469821167, 'hidden_dim': 256}. Best is trial 0 with value: 0.6027663431071133.
[I 2026-10-25 21:24:08,254] Trial 1 finished with value: 0.6181371439399058 and parameters: {'learning_rate': 0.0002930417045619916, 'weight_decay': 8.899066637291121e-06, 'dropout': 0.30948682666820515, 'hidden_dim': 512}. Best is trial 0 with value: 0.6027663431071133.
[I 2026-10-25 22:04:49,410] Trial 2 finished with value: 0.645869956195699 and parameters: {'learning_rate': 1.7940332865835286e-05, 'weight_decay': 0.004620803792507943, 'dropout': 0.5476660039948422, 'hidden_dim': 256}. Best is trial 0 with value: 0.6027663431071133.
[I 2026-10-25 22:20:48,560] Trial 3 finished with value: 0.6117116715866706 and parameters: {'learning_rat

Melhores hiperparâmetros:
{'learning_rate': 0.0009459745995761232, 'weight_decay': 4.530955888267028e-06, 'dropout': 0.21570375752253387, 'hidden_dim': 256}
Melhor val loss:
0.5639777655546376
